# 04 — Model Optimization

This notebook continues directly from **Notebook 03 — Machine Learning Models**.

Notebook 03 established an identity-aware train–test split, trained the baseline
regression models, and saved all information needed to reproduce that split.
This notebook keeps the test set unchanged and optimizes the three strongest
non-linear model families:

- Random Forest
- Gradient Boosting
- XGBoost

Hyperparameters are selected only on the training set using
**identity-aware cross-validation**. The untouched test set is used once for the
final comparison with the corresponding baseline models.

## Research purpose

The objective is not merely to obtain a lower error, but to determine whether
hyperparameter tuning produces a stable and practically meaningful improvement
in the prediction of CR-FIQA scores.

## Inputs from Notebook 03

- `results/03_ml_models/tables/model_comparison.csv`
- `results/03_ml_models/tables/identity_aware_split.csv`
- `results/03_ml_models/run_metadata.json`
- the merged dataset created in Notebook 01

## Main outputs

- tuned model pipelines
- best hyperparameters
- cross-validation and test-set metrics
- baseline-versus-tuned comparison
- predictions and residuals for later analyses
- feature importance of the final optimized model

## 1. Shared project setup

In [ ]:
# Locate and run the shared setup notebook used by Notebooks 00–03.

from pathlib import Path

setup_candidates = [
    Path.cwd() / "00_colab_setup.ipynb",
    Path.cwd() / "notebooks" / "00_colab_setup.ipynb",
    Path.cwd().parent / "notebooks" / "00_colab_setup.ipynb",
    Path("/content/drive/MyDrive/FIQA_Project/notebooks/00_colab_setup.ipynb"),
    Path("/content/drive/MyDrive/FIQA_Project/00_colab_setup.ipynb"),
]

SETUP_NOTEBOOK = next(
    (path for path in setup_candidates if path.exists()),
    None,
)

if SETUP_NOTEBOOK is None:
    checked_paths = "\n".join(f"- {path}" for path in setup_candidates)
    raise FileNotFoundError(
        "00_colab_setup.ipynb could not be found.\n"
        "Keep the notebooks in the same notebooks/ folder or update "
        "setup_candidates.\n\n"
        f"Checked:\n{checked_paths}"
    )

print(f"Running setup notebook: {SETUP_NOTEBOOK}")
get_ipython().run_line_magic("run", f'"{SETUP_NOTEBOOK}"')

## 2. Imports and configuration

In [ ]:
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "XGBoost is required. Install it with `pip install xgboost` "
        "and rerun this cell."
    ) from exc

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
N_CV_SPLITS = 5
TARGET = "cr_fiqa_score"
IDENTITY_COLUMN = "cls"

# Use "quick" while developing and "full" for the final project run.
RUN_MODE = "quick"

SEARCH_ITERATIONS = {
    "quick": {
        "Random Forest": 12,
        "Gradient Boosting": 12,
        "XGBoost": 12,
    },
    "full": {
        "Random Forest": 60,
        "Gradient Boosting": 60,
        "XGBoost": 60,
    },
}

if RUN_MODE not in SEARCH_ITERATIONS:
    raise ValueError("RUN_MODE must be either 'quick' or 'full'.")

print(f"Run mode: {RUN_MODE}")
print(f"Identity-aware CV folds: {N_CV_SPLITS}")

## 3. Input and output paths

In [ ]:
MERGED_FILE = PROJECT_PATH / "diveface_fiqa_merged.csv"

NOTEBOOK_03_RESULTS = PROJECT_PATH / "results" / "03_ml_models"
NOTEBOOK_03_TABLES = NOTEBOOK_03_RESULTS / "tables"

BASELINE_RESULTS_FILE = NOTEBOOK_03_TABLES / "model_comparison.csv"
SPLIT_FILE = NOTEBOOK_03_TABLES / "identity_aware_split.csv"
RUN_METADATA_FILE = NOTEBOOK_03_RESULTS / "run_metadata.json"

RESULTS_PATH = PROJECT_PATH / "results" / "04_model_optimization"
FIGURES_PATH = RESULTS_PATH / "figures"
TABLES_PATH = RESULTS_PATH / "tables"
MODELS_PATH = PROJECT_PATH / "models" / "04_model_optimization"

for path in [RESULTS_PATH, FIGURES_PATH, TABLES_PATH, MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

required_files = [
    MERGED_FILE,
    BASELINE_RESULTS_FILE,
    SPLIT_FILE,
    RUN_METADATA_FILE,
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    missing_text = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Required inputs are missing. Run Notebooks 01 and 03 first.\n"
        f"{missing_text}"
    )

print(f"Dataset:             {MERGED_FILE}")
print(f"Notebook 03 results: {NOTEBOOK_03_RESULTS}")
print(f"Notebook 04 results: {RESULTS_PATH}")

## 4. Load the Notebook 03 configuration and reproduce its exact split

The train–test split is **not generated again**. Instead, the saved row-level
split assignment from Notebook 03 is joined back to the merged dataset. This
ensures that baseline and tuned models are evaluated on exactly the same images
and identities.

In [ ]:
with RUN_METADATA_FILE.open("r", encoding="utf-8") as file:
    run_metadata = json.load(file)

feature_columns = run_metadata["feature_columns"]
baseline_results = pd.read_csv(BASELINE_RESULTS_FILE)
split_information = pd.read_csv(SPLIT_FILE)
df = pd.read_csv(MERGED_FILE)

required_dataset_columns = set(
    feature_columns + [TARGET, IDENTITY_COLUMN, "index"]
)
missing_columns = sorted(required_dataset_columns.difference(df.columns))

if missing_columns:
    raise KeyError(
        "The merged dataset is missing required columns: "
        f"{missing_columns}"
    )

required_split_columns = {
    "index",
    IDENTITY_COLUMN,
    "source_dataframe_index",
    "split",
}
missing_split_columns = sorted(
    required_split_columns.difference(split_information.columns)
)

if missing_split_columns:
    raise KeyError(
        "The Notebook 03 split file is missing required columns: "
        f"{missing_split_columns}"
    )

# Reconstruct the same complete-case modeling table used in Notebook 03.
model_df = df[
    ["index", IDENTITY_COLUMN, TARGET] + feature_columns
].copy()

numeric_features = [
    feature
    for feature in feature_columns
    if feature != "group"
]

for column in numeric_features + [TARGET]:
    model_df[column] = pd.to_numeric(model_df[column], errors="coerce")

model_df = model_df.dropna(
    subset=[IDENTITY_COLUMN, TARGET] + feature_columns
).copy()

model_df["source_dataframe_index"] = model_df.index

data_with_split = model_df.merge(
    split_information[
        ["source_dataframe_index", "split"]
    ],
    on="source_dataframe_index",
    how="inner",
    validate="one_to_one",
)

if len(data_with_split) != len(split_information):
    raise RuntimeError(
        "The saved split could not be matched exactly to the current dataset. "
        "Confirm that Notebook 03 and Notebook 04 use the same merged CSV."
    )

train_df = data_with_split.loc[
    data_with_split["split"] == "train"
].copy()

test_df = data_with_split.loc[
    data_with_split["split"] == "test"
].copy()

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()
y_train = train_df[TARGET].copy()
y_test = test_df[TARGET].copy()

train_identities = train_df[IDENTITY_COLUMN].copy()
test_identities = test_df[IDENTITY_COLUMN].copy()

identity_overlap = set(train_identities).intersection(set(test_identities))

if identity_overlap:
    raise RuntimeError(
        f"Identity leakage detected for {len(identity_overlap)} identities."
    )

expected_train_rows = run_metadata.get("training_rows")
expected_test_rows = run_metadata.get("test_rows")

if expected_train_rows is not None and len(X_train) != expected_train_rows:
    raise RuntimeError(
        f"Training row count differs from Notebook 03: "
        f"{len(X_train)} != {expected_train_rows}"
    )

if expected_test_rows is not None and len(X_test) != expected_test_rows:
    raise RuntimeError(
        f"Test row count differs from Notebook 03: "
        f"{len(X_test)} != {expected_test_rows}"
    )

print(f"Training rows:       {len(X_train):,}")
print(f"Test rows:           {len(X_test):,}")
print(f"Training identities: {train_identities.nunique():,}")
print(f"Test identities:     {test_identities.nunique():,}")
print("Identity overlap:    0")

## 5. Feature groups and preprocessing

In [ ]:
continuous_features = [
    "age",
    "smile",
    "moustache",
    "beard",
    "sideburns",
    "head_roll",
    "head_yaw",
    "head_pitch",
    "blur",
    "exposure",
    "noise",
]

binary_features = [
    "mask",
    "headWear",
    "glasses",
    "eye_makeup",
    "lip_makeup",
    "forehead_occluded",
    "eye_occluded",
    "mouth_occluded",
]

categorical_features = ["group"]

configured_features = (
    continuous_features
    + binary_features
    + categorical_features
)

if configured_features != feature_columns:
    raise ValueError(
        "The feature configuration differs from Notebook 03. "
        "Keep both notebooks synchronized before tuning."
    )

preprocessor_template = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False,
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print(f"Continuous features:  {len(continuous_features)}")
print(f"Binary features:      {len(binary_features)}")
print(f"Categorical features: {len(categorical_features)}")

## 6. Identity-aware cross-validation

`GroupKFold` keeps all images belonging to one identity in the same fold.
Preprocessing remains inside each model pipeline, so the scaler and encoder are
fitted separately within every training fold.

In [ ]:
if train_identities.nunique() < N_CV_SPLITS:
    raise ValueError(
        "The number of training identities is smaller than the requested "
        "number of cross-validation folds."
    )

group_cv = GroupKFold(n_splits=N_CV_SPLITS)

scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
}

## 7. Evaluation helper

In [ ]:
def regression_metrics(y_true, y_pred):
    """Calculate the project regression metrics."""
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
    }


def evaluate_pipeline(model_name, fitted_pipeline):
    """Evaluate a fitted pipeline on the unchanged train and test sets."""
    train_predictions = fitted_pipeline.predict(X_train)
    test_predictions = fitted_pipeline.predict(X_test)

    train_metrics = regression_metrics(y_train, train_predictions)
    test_metrics = regression_metrics(y_test, test_predictions)

    result = {
        "model": model_name,
        "train_mae": train_metrics["mae"],
        "test_mae": test_metrics["mae"],
        "train_rmse": train_metrics["rmse"],
        "test_rmse": test_metrics["rmse"],
        "train_r2": train_metrics["r2"],
        "test_r2": test_metrics["r2"],
    }

    return result, train_predictions, test_predictions

## 8. Models and hyperparameter search spaces

The ranges cover the most relevant complexity, regularization, sampling, and
learning-rate parameters for each model family. `RandomizedSearchCV` is used
because an exhaustive Cartesian grid would be unnecessarily expensive.

In [ ]:
model_definitions = {
    "Random Forest": {
        "estimator": RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "parameters": {
            "regressor__n_estimators": randint(200, 1001),
            "regressor__max_depth": [None, 5, 10, 15, 20, 30, 40],
            "regressor__min_samples_split": randint(2, 21),
            "regressor__min_samples_leaf": randint(1, 11),
            "regressor__max_features": [
                "sqrt",
                "log2",
                0.5,
                0.7,
                1.0,
            ],
            "regressor__bootstrap": [True, False],
        },
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingRegressor(
            random_state=RANDOM_STATE,
            loss="squared_error",
        ),
        "parameters": {
            "regressor__n_estimators": randint(100, 701),
            "regressor__learning_rate": loguniform(0.01, 0.2),
            "regressor__max_depth": randint(2, 7),
            "regressor__min_samples_split": randint(2, 21),
            "regressor__min_samples_leaf": randint(1, 11),
            "regressor__subsample": uniform(0.6, 0.4),
            "regressor__max_features": [
                None,
                "sqrt",
                "log2",
                0.5,
                0.8,
            ],
        },
    },
    "XGBoost": {
        "estimator": XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
        ),
        "parameters": {
            "regressor__n_estimators": randint(200, 1201),
            "regressor__learning_rate": loguniform(0.01, 0.2),
            "regressor__max_depth": randint(2, 9),
            "regressor__min_child_weight": randint(1, 11),
            "regressor__subsample": uniform(0.6, 0.4),
            "regressor__colsample_bytree": uniform(0.6, 0.4),
            "regressor__reg_alpha": loguniform(1e-5, 1.0),
            "regressor__reg_lambda": loguniform(0.1, 10.0),
        },
    },
}

## 9. Hyperparameter tuning

In [ ]:
search_objects = {}
tuned_pipelines = {}
tuning_summary_rows = []

for model_name, definition in model_definitions.items():
    print("\n" + "=" * 80)
    print(f"Optimizing: {model_name}")
    print("=" * 80)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor_template)),
            ("regressor", definition["estimator"]),
        ]
    )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=definition["parameters"],
        n_iter=SEARCH_ITERATIONS[RUN_MODE][model_name],
        scoring=scoring,
        refit="RMSE",
        cv=group_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
        return_train_score=True,
        error_score="raise",
    )

    start_time = time.perf_counter()

    search.fit(
        X_train,
        y_train,
        groups=train_identities,
    )

    elapsed_minutes = (time.perf_counter() - start_time) / 60
    best_index = search.best_index_

    search_objects[model_name] = search
    tuned_pipelines[model_name] = search.best_estimator_

    tuning_summary_rows.append(
        {
            "model": model_name,
            "best_cv_rmse": -search.cv_results_[
                "mean_test_RMSE"
            ][best_index],
            "cv_rmse_sd": search.cv_results_[
                "std_test_RMSE"
            ][best_index],
            "best_cv_mae": -search.cv_results_[
                "mean_test_MAE"
            ][best_index],
            "best_cv_r2": search.cv_results_[
                "mean_test_R2"
            ][best_index],
            "search_minutes": elapsed_minutes,
            "candidates_tested": len(search.cv_results_["params"]),
        }
    )

tuning_summary = (
    pd.DataFrame(tuning_summary_rows)
    .sort_values("best_cv_rmse")
    .reset_index(drop=True)
)

display(tuning_summary.round(4))

## 10. Evaluate tuned models on the unchanged test set

In [ ]:
tuned_result_rows = []
train_predictions_by_model = {}
test_predictions_by_model = {}

for model_name, fitted_pipeline in tuned_pipelines.items():
    result, train_predictions, test_predictions = evaluate_pipeline(
        model_name,
        fitted_pipeline,
    )

    tuned_result_rows.append(result)
    train_predictions_by_model[model_name] = train_predictions
    test_predictions_by_model[model_name] = test_predictions

tuned_results = (
    pd.DataFrame(tuned_result_rows)
    .sort_values("test_rmse")
    .reset_index(drop=True)
)

display(tuned_results.round(4))

## 11. Baseline-versus-tuned comparison

Only corresponding model families are compared. The baseline values are read
directly from Notebook 03 rather than recalculated.

In [ ]:
optimized_model_names = list(model_definitions)

required_baseline_columns = {
    "model",
    "train_mae",
    "test_mae",
    "train_rmse",
    "test_rmse",
    "train_r2",
    "test_r2",
}

missing_baseline_columns = sorted(
    required_baseline_columns.difference(baseline_results.columns)
)

if missing_baseline_columns:
    raise KeyError(
        "Notebook 03 model_comparison.csv is missing columns: "
        f"{missing_baseline_columns}"
    )

baseline_selected = baseline_results.loc[
    baseline_results["model"].isin(optimized_model_names),
    sorted(required_baseline_columns),
].copy()

if set(baseline_selected["model"]) != set(optimized_model_names):
    missing_models = sorted(
        set(optimized_model_names).difference(baseline_selected["model"])
    )
    raise KeyError(
        "Notebook 03 baseline results are missing model(s): "
        f"{missing_models}"
    )

baseline_selected["version"] = "Baseline"
tuned_selected = tuned_results.copy()
tuned_selected["version"] = "Tuned"

metric_columns = [
    "train_mae",
    "test_mae",
    "train_rmse",
    "test_rmse",
    "train_r2",
    "test_r2",
]

baseline_vs_tuned = pd.concat(
    [
        baseline_selected[["model", "version"] + metric_columns],
        tuned_selected[["model", "version"] + metric_columns],
    ],
    ignore_index=True,
).sort_values(["model", "version"])

display(baseline_vs_tuned.round(4))

In [ ]:
baseline_lookup = baseline_selected.set_index("model")
tuned_lookup = tuned_results.set_index("model")

improvement_rows = []

for model_name in optimized_model_names:
    baseline_row = baseline_lookup.loc[model_name]
    tuned_row = tuned_lookup.loc[model_name]

    improvement_rows.append(
        {
            "model": model_name,
            "baseline_test_rmse": baseline_row["test_rmse"],
            "tuned_test_rmse": tuned_row["test_rmse"],
            "rmse_improvement_percent": (
                (
                    baseline_row["test_rmse"]
                    - tuned_row["test_rmse"]
                )
                / baseline_row["test_rmse"]
                * 100
            ),
            "baseline_test_mae": baseline_row["test_mae"],
            "tuned_test_mae": tuned_row["test_mae"],
            "mae_improvement_percent": (
                (
                    baseline_row["test_mae"]
                    - tuned_row["test_mae"]
                )
                / baseline_row["test_mae"]
                * 100
            ),
            "baseline_test_r2": baseline_row["test_r2"],
            "tuned_test_r2": tuned_row["test_r2"],
            "r2_change": (
                tuned_row["test_r2"]
                - baseline_row["test_r2"]
            ),
        }
    )

improvement_results = pd.DataFrame(improvement_rows)
display(improvement_results.round(4))

## 12. Generalization and overfitting check

In [ ]:
generalization_results = tuned_results[
    [
        "model",
        "train_rmse",
        "test_rmse",
        "train_r2",
        "test_r2",
    ]
].merge(
    tuning_summary[
        [
            "model",
            "best_cv_rmse",
            "cv_rmse_sd",
            "best_cv_r2",
        ]
    ],
    on="model",
    how="left",
)

generalization_results["train_test_rmse_gap"] = (
    generalization_results["test_rmse"]
    - generalization_results["train_rmse"]
)

generalization_results["cv_test_rmse_gap"] = (
    generalization_results["test_rmse"]
    - generalization_results["best_cv_rmse"]
)

display(generalization_results.round(4))

## 13. Select the final optimized model

The final optimized model is selected by the lowest test RMSE after all
hyperparameters have been determined exclusively through training-set
cross-validation. Because the same test set is used to compare the three tuned
families, cross-validation stability and train–test gaps must also be reported.

In [ ]:
final_model_name = tuned_results.loc[0, "model"]
final_model = tuned_pipelines[final_model_name]
final_test_predictions = test_predictions_by_model[final_model_name]

final_result = tuned_results.loc[
    tuned_results["model"] == final_model_name
].iloc[0]

print(f"Final optimized model: {final_model_name}")
print(f"Test RMSE: {final_result['test_rmse']:.4f}")
print(f"Test MAE:  {final_result['test_mae']:.4f}")
print(f"Test R²:   {final_result['test_r2']:.4f}")

## 14. Diagnostic plots

In [ ]:
rmse_plot_data = baseline_vs_tuned.pivot(
    index="model",
    columns="version",
    values="test_rmse",
)

ax = rmse_plot_data.plot(
    kind="bar",
    figsize=(10, 6),
)

ax.set_title("Baseline vs. Tuned Test RMSE")
ax.set_xlabel("Model")
ax.set_ylabel("Test RMSE")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "baseline_vs_tuned_test_rmse.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
minimum_value = min(
    y_test.min(),
    final_test_predictions.min(),
)
maximum_value = max(
    y_test.max(),
    final_test_predictions.max(),
)

plt.figure(figsize=(7, 7))
plt.scatter(y_test, final_test_predictions, alpha=0.4)
plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--",
)
plt.xlabel("Actual CR-FIQA Score")
plt.ylabel("Predicted CR-FIQA Score")
plt.title(f"Actual vs. Predicted — {final_model_name}")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "final_model_actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
final_residuals = y_test.to_numpy() - final_test_predictions

plt.figure(figsize=(8, 5))
plt.scatter(final_test_predictions, final_residuals, alpha=0.4)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted CR-FIQA Score")
plt.ylabel("Residual: Actual − Predicted")
plt.title(f"Residual Plot — {final_model_name}")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "final_model_residual_plot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(final_residuals, bins=40)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title(f"Residual Distribution — {final_model_name}")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "final_model_residual_distribution.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 15. Feature importance of the final optimized model

All optimized candidates are tree-based and expose impurity- or gain-based
feature importance. These values describe the model's internal predictive use
of features; they do not establish causality.

In [ ]:
final_preprocessor = final_model.named_steps["preprocessor"]
final_estimator = final_model.named_steps["regressor"]

processed_feature_names = final_preprocessor.get_feature_names_out()

if not hasattr(final_estimator, "feature_importances_"):
    raise AttributeError(
        "The selected model does not expose feature_importances_."
    )

final_feature_importance = (
    pd.DataFrame(
        {
            "feature": processed_feature_names,
            "importance": final_estimator.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(final_feature_importance.head(20).round(4))

In [ ]:
top_features = (
    final_feature_importance
    .head(15)
    .sort_values("importance", ascending=True)
)

plt.figure(figsize=(9, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title(f"Top Feature Importances — {final_model_name}")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "final_model_feature_importance.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 16. Save optimized models and results

In [ ]:
model_file_names = {
    "Random Forest": "tuned_random_forest.joblib",
    "Gradient Boosting": "tuned_gradient_boosting.joblib",
    "XGBoost": "tuned_xgboost.joblib",
}

for model_name, fitted_pipeline in tuned_pipelines.items():
    model_file = MODELS_PATH / model_file_names[model_name]
    joblib.dump(fitted_pipeline, model_file)

joblib.dump(
    final_model,
    MODELS_PATH / "best_tuned_model.joblib",
)

print("Saved tuned model pipelines.")

In [ ]:
tuning_summary.to_csv(
    TABLES_PATH / "tuning_summary.csv",
    index=False,
)

tuned_results.to_csv(
    TABLES_PATH / "tuned_model_comparison.csv",
    index=False,
)

baseline_vs_tuned.to_csv(
    TABLES_PATH / "baseline_vs_tuned_comparison.csv",
    index=False,
)

improvement_results.to_csv(
    TABLES_PATH / "model_improvements.csv",
    index=False,
)

generalization_results.to_csv(
    TABLES_PATH / "generalization_analysis.csv",
    index=False,
)

final_feature_importance.to_csv(
    TABLES_PATH / "best_model_feature_importance.csv",
    index=False,
)

print("Saved optimization tables.")

In [ ]:
def to_json_serializable(value):
    """Convert NumPy values and nested containers for JSON output."""
    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            key: to_json_serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_json_serializable(item)
            for item in value
        ]

    return value


best_hyperparameters = {
    model_name: to_json_serializable(search.best_params_)
    for model_name, search in search_objects.items()
}

with (
    RESULTS_PATH / "best_hyperparameters.json"
).open("w", encoding="utf-8") as file:
    json.dump(best_hyperparameters, file, indent=2)

print("Saved best hyperparameters.")

In [ ]:
prediction_results = test_df[
    [
        "index",
        IDENTITY_COLUMN,
        "group",
        TARGET,
        "source_dataframe_index",
    ]
].copy()

prediction_results = prediction_results.rename(
    columns={TARGET: "actual_cr_fiqa_score"}
)

for model_name, model_predictions in test_predictions_by_model.items():
    safe_name = model_name.lower().replace(" ", "_")

    prediction_results[f"{safe_name}_prediction"] = model_predictions
    prediction_results[f"{safe_name}_residual"] = (
        prediction_results["actual_cr_fiqa_score"].to_numpy()
        - model_predictions
    )

prediction_results.to_csv(
    TABLES_PATH / "tuned_model_predictions.csv",
    index=False,
)

print("Saved tuned predictions and residuals.")

In [ ]:
optimization_summary = {
    "run_mode": RUN_MODE,
    "random_state": RANDOM_STATE,
    "cv_folds": N_CV_SPLITS,
    "selection_metric": "test_rmse",
    "final_model": final_model_name,
    "final_test_metrics": {
        "rmse": float(final_result["test_rmse"]),
        "mae": float(final_result["test_mae"]),
        "r2": float(final_result["test_r2"]),
    },
    "feature_columns": feature_columns,
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "training_identities": int(train_identities.nunique()),
    "test_identities": int(test_identities.nunique()),
}

with (
    RESULTS_PATH / "model_optimization_summary.json"
).open("w", encoding="utf-8") as file:
    json.dump(optimization_summary, file, indent=2)

print("Saved optimization summary.")

## 17. Saved-file summary

In [ ]:
print("Notebook 04 result files:")
for file_path in sorted(RESULTS_PATH.rglob("*")):
    if file_path.is_file():
        print("-", file_path.relative_to(RESULTS_PATH))

print("\nOptimized model files:")
for file_path in sorted(MODELS_PATH.glob("*.joblib")):
    print("-", file_path.name)

## 18. Interpretation and connection to later notebooks

Before reporting an optimized model as an improvement, examine:

1. whether its test RMSE and MAE improved over the Notebook 03 baseline;
2. whether the improvement is large enough to be practically meaningful;
3. whether cross-validation RMSE is close to test RMSE;
4. whether the train–test gap suggests overfitting;
5. whether the selected features remain interpretable and stable.

The following artifacts support the next stages of the project:

- `models/04_model_optimization/best_tuned_model.joblib`  
  Complete final pipeline for prediction and explainability.

- `results/04_model_optimization/tables/tuned_model_predictions.csv`  
  Test-set predictions, residuals, demographic group, and identity information.

- `results/04_model_optimization/tables/best_model_feature_importance.csv`  
  Model-based feature ranking for comparison with correlation, regression, and
  later explainability analyses.

This notebook focuses only on predictive optimization. Statistical association,
demographic consistency, and model explanation are handled separately in the
following notebooks to avoid mixing distinct research questions.